# sheets_to_banana

Converts samba percursion notes in custom table-based format from Google Sheets into a [BananaDrum](https://www.bannadrum.net) link.

In [ ]:
#@title { display-mode: "form", run: "auto" }
#@markdown Before running this converter, open the Google Sheets file, navigate to the worksheet that contains the samba notes and copy the link from the address bar.
#@markdown (the Google Sheets file must be set to [Anyone with the link can view](https://support.google.com/drive/answer/2494822?hl=en&co=GENIE.Platform%3DDesktop#zippy=%2Cshare-a-single-file%2Cshare-a-file-or-folder-publicly)).
#@markdown
#@markdown Then follow these steps:
#@markdown
#@markdown **Step 1) (mandatory)** Paste the copied link below in the field sheets_link.
#@markdown
#@markdown **Step 2) (optional)** If the worksheet contains more than one break, you may choose to convert all or a single break (1 = first break, 2 = second, etc.). **Leave this field empty** to generate links for **all breaks**.
#@markdown
#@markdown **Step 3) (optional)** You may choose the playback speed of the Bananadrum link in BPM . (default = 110).
#@markdown
#@markdown **Step 4) (optional)** For test purposes only: You may choose to run a test version of the conversion tool by entering the corresponding branch name **Leave this field empty to run the standard version**
#@markdown Click ▶ to generate the BananaDrum link.  
#@markdown Click ▶ to generate the BananaDrum link.  

sheets_link = "" #@param {type:"string", placeholder:"Paste the Google Sheets link here"}
break_number = "" #@param {type:"string", placeholder:"Leave empty for all breaks, or enter a number (e.g. 1)"}
tempo = "110" #@param {type:"string", placeholder:"e.g. 110"}
install_branch = "" #@param {type:"string", placeholder:"Advanced - leave empty to install from main"}

import importlib

# Branch to install sheets_to_banana from. Normal users leave install_branch
# empty and get "main". To test in-progress work, open this notebook from a
# branch's Colab URL and type that branch's name into the install_branch field.
_branch = install_branch.strip() or "main"
if importlib.util.find_spec("sheets_to_banana") is None or globals().get("_installed_branch") != _branch:
    print(f"⌛ Installing sheets_to_banana from branch '{_branch}' — this may take about 30 seconds...")
    import subprocess, shutil
    if shutil.which("uv") is None:
        subprocess.run(["pip", "install", "uv", "-q"], check=True, capture_output=True)
    _install = subprocess.run(
        ["uv", "pip", "install", "--reinstall-package", "sheets_to_banana",
         f"git+https://github.com/bruno-ritmista/samba.git@{_branch}#subdirectory=sheets_to_banana",
         "--system", "-q"],
        capture_output=True
    )
    if _install.returncode != 0:
        print("❌ Installation failed:\n" + _install.stderr.decode())
        raise SystemExit(1)
    _installed_branch = _branch
    print("✅ Conversion tool ready.")

import threading, time
if not any(t.name == "colab_keepalive" for t in threading.enumerate()):
    def _keep_alive():
        while True:
            time.sleep(60)
    _t = threading.Thread(target=_keep_alive, name="colab_keepalive", daemon=True)
    _t.start()

import logging as _logging
import re

def _setup_notebook_logging():
    class _FriendlyFormatter(_logging.Formatter):
        def format(self, record):
            if record.levelno == _logging.WARNING:
                return f"⚠️  {record.getMessage()}"
            return record.getMessage()

    _h = _logging.StreamHandler()
    _h.setFormatter(_FriendlyFormatter())
    _pkg = _logging.getLogger("sheets_to_banana")
    _pkg.handlers.clear()
    _pkg.addHandler(_h)
    _pkg.propagate = False
    _pkg.setLevel(_logging.WARNING)

_setup_notebook_logging()

def _clean_break_name(name):
    cleaned = re.sub(r'\s*\(.*\)\s*$', '', name).strip()
    cleaned = re.sub(r'(?i)\s*[-\s_]*banana[-\s_]*drum[-\s_]*', '', cleaned).strip(' \t-_')
    return cleaned

def _run():
    brk_num = int(break_number) if break_number else None
    val = sheets_link.strip()

    if not val:
        print("Paste a Google Sheets link in the field above, then run this cell.")
        return

    if re.search(r'/spreadsheets/d/([a-zA-Z0-9_-]+)', val):
        full_url = val
    elif re.fullmatch(r'[a-zA-Z0-9_-]+', val):
        full_url = f"https://docs.google.com/spreadsheets/d/{val}"
    else:
        print("❌ That doesn't look like a Google Sheets link. "
              "Please copy the link from the sheet and try again.")
        return

    tempo_clean = re.sub(r'(?i)\s*bpm\s*', '', str(tempo)).strip()
    try:
        tempo_int = int(tempo_clean)
    except ValueError:
        print("❌ Tempo must be a number (e.g. 120 or \"120 bpm\"). Please fix the Tempo field and try again.")
        return

    try:
        from sheets_to_banana.fetch import fetch_csv
        from sheets_to_banana.parse import parse_sheet
        from sheets_to_banana.mapping import map_break, trim_empty_bars
        from sheets_to_banana.encode import encode_url
    except ImportError:
        print("❌ The conversion tool failed to load. Please reload the page and try again.")
        return

    try:
        csv_text = fetch_csv(full_url)
    except Exception:
        print("❌ Could not read your Google Sheet. "
              "Please check the link and try again.")
        return

    breaks = parse_sheet(csv_text)
    if not breaks:
        print("❌ No breaks found in the sheet. "
              "Please check that the sheet has the expected format.")
        return

    if brk_num is None:
        selected = list(enumerate(breaks, start=1))
    elif brk_num < 1 or brk_num > len(breaks):
        print(f"❌ Break {break_number} not found. "
              f"This sheet has {len(breaks)} break(s): 1–{len(breaks)}. "
              f"Leave the field empty to convert all breaks.")
        return
    else:
        selected = [(brk_num, breaks[brk_num - 1])]

    for num, brk in selected:
        tracks = map_break(brk)
        if not tracks:
            print(f"⚠️  Break {num} \"{brk.name}\" has no recognised instruments — skipped.")
            continue

        trim = trim_empty_bars(tracks)
        if trim.all_empty:
            print(f"⚠️  Break {num} \"{brk.name}\" — all bars empty, skipped.")
            continue
        tracks = trim.tracks

        n_bars = max(len(t.notes) for t in tracks) // 16
        clean_name = _clean_break_name(brk.name)
        url = encode_url(tracks, tempo=tempo_int, n_bars=n_bars, title=clean_name)

        print(f"✅ BananaDrum link for \"{clean_name}\" "
              f"({n_bars} bars, {len(tracks)} instruments):")
        print()
        print("\U0001f517 " + url)
        print()

_run()